## problem 1 (a)

Team: Lea Azar, Pir Mujtaba , Ekin Kaya, Hamza Shahzad

In [2]:
import pandas as pd

In [3]:
spy=pd.read_csv("data/SPY_data.csv")
cpi=pd.read_csv("data/release_dates_10.csv")
spy=spy.rename(columns={ "DlyCalDt": "date",
    "DlyRet": "return"})
print(spy.head())
print(cpi.head())

   PERMNO  HdrCUSIP Ticker  PERMCO        date    return
0   84398  78462F10    SPY   46699  2020-01-02  0.009352
1   84398  78462F10    SPY   46699  2020-01-03 -0.007572
2   84398  78462F10    SPY   46699  2020-01-06  0.003815
3   84398  78462F10    SPY   46699  2020-01-07 -0.002812
4   84398  78462F10    SPY   46699  2020-01-08  0.005330
  release_date
0   2021-01-13
1   2021-02-10
2   2021-03-10
3   2021-04-13
4   2021-05-12


In [4]:
spy["date"] = pd.to_datetime(spy["date"])
cpi["release_date"] = pd.to_datetime(cpi["release_date"])
# Sample period is 2021 onward (the question asks for 2021-2026)
spy = spy[spy["date"] >= "2021-01-01"].reset_index(drop=True)
print(spy.dtypes)
print(cpi.dtypes)

PERMNO               int64
HdrCUSIP            object
Ticker              object
PERMCO               int64
date        datetime64[ns]
return             float64
dtype: object
release_date    datetime64[ns]
dtype: object


In [5]:
spy["return"]=pd.to_numeric(spy["return"], errors='coerce')
print ("missing values:", spy["return"].isnull().sum())

missing values: 0


In [6]:
print("Duplicate SPY dates:", spy["date"].duplicated().sum())
print("Duplicate CPI dates:", cpi["release_date"].duplicated().sum())

Duplicate SPY dates: 0
Duplicate CPI dates: 0


In [7]:
data = spy.merge(
    cpi,
    left_on="date",
    right_on="release_date",
    how="left",
    validate="one_to_one",
    indicator=True
)
data.to_csv("data/analysis_data.csv", index=False)

In [8]:

data["cpi_day"]=(data["_merge"]=="both").astype(int)
print(data["cpi_day"].value_counts())

cpi_day
0    1196
1      59
Name: count, dtype: int64


The maximum allowed date from WRDS was 2025 hence why 9 cpi dates are unmatched

In [9]:
group_stats = data.groupby("cpi_day")["return"].agg(
    ["count", "mean", "std"]
)

print(group_stats)

         count      mean       std
cpi_day                           
0         1196  0.000563  0.010553
1           59  0.001181  0.014776


Now we will compute the regression on returns: 


Return_t =alpha+ beta* CpiDay_t +epsilon_t

In [10]:
import statsmodels.api as sm


In [11]:
reg_data = data[["return", "cpi_day"]].dropna().copy()
y = reg_data["return"]

X = reg_data[["cpi_day"]]
X = sm.add_constant(X)
model = sm.OLS(y, X).fit(cov_type="HC1")

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                 return   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.001
Method:                 Least Squares   F-statistic:                    0.1021
Date:                Sun, 04 Oct 2026   Prob (F-statistic):              0.749
Time:                        23:30:34   Log-Likelihood:                 3904.9
No. Observations:                1255   AIC:                            -7806.
Df Residuals:                    1253   BIC:                            -7795.
Df Model:                           1                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0006      0.000      1.844      0.0

In [12]:
ordinary_mean = reg_data.loc[
    reg_data["cpi_day"] == 0, "return"
].mean()

cpi_mean = reg_data.loc[
    reg_data["cpi_day"] == 1, "return"
].mean()

difference = cpi_mean - ordinary_mean

print("Ordinary-day mean:", ordinary_mean * 100, "%")
print("CPI-day mean:", cpi_mean * 100, "%")
print("Difference:", difference * 100, "percentage points")
print("Regression coefficient:", model.params["cpi_day"])

Ordinary-day mean: 0.05629632107023411 %
CPI-day mean: 0.11806440677966101 %
Difference: 0.061768085709426906 percentage points
Regression coefficient: 0.0006176808570942687


In [13]:
beta = model.params["cpi_day"]
two_sided_p = model.pvalues["cpi_day"]

if beta > 0:
    one_sided_p = two_sided_p / 2
else:
    one_sided_p = 1 - two_sided_p / 2

print("CPI-day coefficient:", beta)
print("Two-sided p-value:", two_sided_p)
print("One-sided p-value:", one_sided_p)

CPI-day coefficient: 0.0006176808570942687
Two-sided p-value: 0.7493278417067855
One-sided p-value: 0.37466392085339273


In [14]:
data.to_csv("data/analysis_data.csv", index=False)

Average SPY returns were approximately 0.118% on CPI announcement days and 0.056% on ordinary trading days. The estimated difference was positive at 0.062 percentage points, but it was not statistically significant using a one-sided test (\(p=0.375\)). Therefore, the sample does not provide sufficient evidence that U.S. stock market returns are higher on CPI announcement days.

We're also curious to see if a longer sample will get better outcome so we ran another regression  on a larger sample the CPI return were even less than ordinary days and the p values also showed that as statistically insignificant 